In [ ]:
%pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib openpyxl beautifulsoup4

%pip install pywin32

In [ ]:
USER_EMAIL = "your_email@gmail.com"

# Gmail search criteria
GMAIL_QUERY = 'newer_than:2y subject:"Introduction - Eric Rosenberg"'

# Maximum number of introduction emails to retrieve
MAX_EMAILS = 500

# Output Excel file
# OUTPUT_FILE = "Recruiter_Introduction_Tracker.xlsx"

# Recruiter name used in the email and LinkedIn templates
RECRUITER_NAME = "Adrian"

In [ ]:
print("Gmail query:", GMAIL_QUERY)

results = gmail.users().messages().list(
    userId="me",
    q=GMAIL_QUERY,
    maxResults=500
).execute()

print("Messages returned by Gmail:", len(results.get("messages", [])))

Follow these instructions and run the code below if you get the Error 

RefreshError: ('invalid_grant: Token has been expired or revoked.', {'error': 'invalid_grant', 'error_description': 'Token has been expired or revoked.'})"


Step 1. Stop the notebook
In Jupyter, go to Kernel → Interrupt if the cell is still running.
Step 2. Delete the old token
In the folder where your notebook and credentials.json are stored:
1. Find the file named token.pickle.
2. Delete it.
3. Do not delete credentials.json.

In [ ]:
import os
import pickle

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# Gmail read-only access
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

# Start with a fresh authentication
creds = None

if os.path.exists("token.pickle"):
    with open("token.pickle", "rb") as token:
        creds = pickle.load(token)

if creds and creds.expired and creds.refresh_token:
    creds.refresh(Request())

if not creds or not creds.valid:
    flow = InstalledAppFlow.from_client_secrets_file(
        "credentials.json",
        SCOPES
    )
    creds = flow.run_local_server(port=0)

    with open("token.pickle", "wb") as token:
        pickle.dump(creds, token)

gmail = build("gmail", "v1", credentials=creds)

print("Successfully connected to Gmail!")

Defines tools for reading and interpreting Gmail messages, and the function to search email for messages that match criteria

In [ ]:
import base64
import re

from bs4 import BeautifulSoup
from email.utils import getaddresses
from datetime import datetime


def decode_email_body(data):

    if not data:
        return ""

    decoded = base64.urlsafe_b64decode(
        data + "=" * (-len(data) % 4)
    )

    return decoded.decode("utf-8", errors="replace")


def get_email_body(payload):

    plain_parts = []
    html_parts = []

    def walk_parts(part):

        mime_type = part.get("mimeType", "")
        body_data = part.get("body", {}).get("data")

        if body_data:

            content = decode_email_body(body_data)

            if mime_type == "text/plain":
                plain_parts.append(content)

            elif mime_type == "text/html":
                html_parts.append(content)

        for child in part.get("parts", []):
            walk_parts(child)

    walk_parts(payload)

    if plain_parts:
        return "\n".join(plain_parts)

    if html_parts:

        soup = BeautifulSoup(
            "\n".join(html_parts),
            "html.parser"
        )

        return soup.get_text(separator="\n", strip=True)

    return ""


def get_header(headers, header_name):

    for header in headers:

        if header["name"].lower() == header_name.lower():
            return header["value"]

    return ""


def retrieve_messages():

    messages = []
    page_token = None

    while len(messages) < MAX_EMAILS:

        results = gmail.users().messages().list(
            userId="me",
            q=GMAIL_QUERY,
            maxResults=min(200, MAX_EMAILS - len(messages)),
            pageToken=page_token
        ).execute()

        messages.extend(results.get("messages", []))

        page_token = results.get("nextPageToken")

        if not page_token:
            break

    return messages




Runs the function to retrieve messages according to criteria in first box. Try not to run this very often

In [ ]:
message_list = retrieve_messages()

print(f"Found {len(message_list)} matching emails.")
print(message_list)


In [ ]:

import json

fetched_messages = []
thread_ids = set()

for i, message_summary in enumerate(message_list, start=1):

    message = gmail.users().messages().get(
        userId="me",
        id=message_summary["id"],
        format="full"
    ).execute()

    fetched_messages.append(message)
    thread_ids.add(message["threadId"])

    if i % 10 == 0:
        print(f"Retrieved {i} emails...")

with open("recruiter_emails.json", "w", encoding="utf-8") as f:
    json.dump(fetched_messages, f, ensure_ascii=False)

print(f"Saved {len(fetched_messages)} emails to recruiter_emails.json")

In [ ]:
# Check which email threads already contain a sent reply
replied_thread_ids = set()

print(f"Checking {len(thread_ids)} unique email threads for replies...")

for i, thread_id in enumerate(thread_ids, start=1):

    thread = gmail.users().threads().get(
        userId="me",
        id=thread_id,
        format="full"
    ).execute()

    thread_has_sent_message = any(
        "SENT" in msg.get("labelIds", [])
        for msg in thread.get("messages", [])
    )

    if thread_has_sent_message:
        replied_thread_ids.add(thread_id)

    if i % 10 == 0:
        print(f"Checked {i} threads...")

# Save the results locally
with open("replied_thread_ids.json", "w", encoding="utf-8") as f:
    json.dump(list(replied_thread_ids), f)

print(f"Found {len(replied_thread_ids)} threads with sent replies.")
print("Saved reply-check results to replied_thread_ids.json")

Functions to get the parts of the name (it pulls from the HI part of the email, unless the name is stored by the google email

In [ ]:
from email.utils import parseaddr

def get_name_parts(display_name, email_address, email_body=""):

    name = (display_name or "").strip()

    # Treat an email address as a missing display name
    if "@" in name:
        name = ""

    # If Gmail doesn't provide a usable name, look after "Hi"
    if not name and email_body:

        import re

        match = re.search(
            r'\bHi[\s,:]+([A-Za-z][A-Za-z\'-]*(?:\s+[A-Za-z][A-Za-z\'-]*)?)',
            email_body,
            re.IGNORECASE
        )

        if match:
            name = match.group(1).strip()

    # Do not use the email address as a name
    if not name:
        return "", ""

    parts = name.split()

    first_name = parts[0]
    last_name = " ".join(parts[1:]) if len(parts) > 1 else ""

    return first_name, last_name




def get_company(email_address):

    domain = email_address.split("@")[-1].lower()

    common_personal_domains = {
        "gmail.com",
        "yahoo.com",
        "outlook.com",
        "hotmail.com",
        "icloud.com",
        "aol.com",
        "live.com"
    }

    if domain in common_personal_domains:
        return ""

    return domain.split(".")[0].replace("-", " ").title()



Extracting the data

In [ ]:
#extracting the data into a json so that it can be used multiplpe times
import json

# Load previously retrieved emails
with open("recruiter_emails.json", "r", encoding="utf-8") as f:

    fetched_messages = json.load(f)

print(f"Loaded {len(fetched_messages)} saved emails.")



contacts_by_email = {}

for message in fetched_messages:

    headers = message["payload"].get("headers", [])

    sender_header = get_header(headers, "From")
    to_header = get_header(headers, "To")
    cc_header = get_header(headers, "Cc")
    subject = get_header(headers, "Subject")

    sender_name, sender_email = parseaddr(sender_header)

    sender_email = sender_email.lower().strip()

    # Collect possible contacts from To and Cc
    recipients = getaddresses([to_header, cc_header])

    candidates = []

    for display_name, email_address in recipients:

        email_address = email_address.lower().strip()

        if not email_address:
            continue

        if email_address == USER_EMAIL.lower():
            continue

        if email_address == sender_email:
            continue

        if email_address.endswith("@gmail.com"):
            continue

        candidates.append((display_name, email_address))

    # Skip emails without an identifiable contact
    if not candidates:
        continue

    body = get_email_body(message["payload"])

    internal_date = int(message.get("internalDate", 0))

    email_date = datetime.fromtimestamp(
        internal_date / 1000
    )

    thread_id = message["threadId"]

    for display_name, contact_email in candidates:

        first_name, last_name = get_name_parts(
            display_name,
            contact_email, 
            body
        )
        print (first_name)
        company = get_company(contact_email)

        
        contact_record = {
            "First Name": first_name,
            "Last Name": last_name,
            "Company": company,
            "Email": contact_email,
            "Email 1": contact_email,
            "Email 2": "",
            "Email 3": "",
            "Introduction Date": email_date,
            "Recruiter Email": sender_email,
            "Original Subject": subject,
            "Internal Date": internal_date,
            "Thread IDs": {thread_id}
        }
        
        if contact_email not in contacts_by_email:
            contacts_by_email[contact_email] = contact_record
        
        else:
            existing = contacts_by_email[contact_email]
            existing["Thread IDs"].add(thread_id)


print(f"Unique contacts found: {len(contacts_by_email)}")
print(contacts_by_email)

In [ ]:
import json

# Load the previously saved reply-check results
with open("replied_thread_ids.json", "r", encoding="utf-8") as f:
    replied_thread_ids = set(json.load(f))

# Identify contacts with at least one thread containing a sent reply
already_replied = set()

for contact_email, contact in contacts_by_email.items():

    if contact["Thread IDs"].intersection(replied_thread_ids):
        already_replied.add(contact_email)

# Remove contacts we've already replied to
contacts_by_email = {
    email: contact
    for email, contact in contacts_by_email.items()
    if email not in already_replied
}

print(f"Contacts already replied to: {len(already_replied)}")
print(f"Contacts remaining for outreach: {len(contacts_by_email)}")

In [ ]:
print(contacts_by_email)

In [ ]:
def create_email_reply(first_name):

    return f"""Hi {first_name},

It's a pleasure to be introduced. I'd welcome the opportunity to meet, share a bit more about my background, and hear your perspective on potential opportunities where my experience could be a good fit.

I'll also send you a LinkedIn connection request.

Thanks, {RECRUITER_NAME}, for making the introduction!

Best,
Eric"""

In [ ]:
def create_linkedin_note(first_name):

    return (
        f"Hi {first_name}, {RECRUITER_NAME} recently introduced us via email. "
        "I'd welcome the opportunity to connect here as well and hope we can find time to speak. "
        "Best, Eric"
    )

In [ ]:
from urllib.parse import quote_plus

import pandas as pd


rows = []

for email_address, contact in contacts_by_email.items():

    first_name = contact["First Name"]

    original_subject = contact["Original Subject"]

    email_subject = (
        f"Re: {original_subject}"
        if original_subject
        else "Introduction"
    )

    email_reply = create_email_reply(first_name)

    linkedin_note = create_linkedin_note(first_name)

    linkedin_search = (
        "https://www.linkedin.com/search/results/people/?keywords="
        + quote_plus(
            f'{first_name} {contact["Last Name"]} {contact["Company"]}'
        )
    )

    rows.append({

        "First Name": first_name,

        "Last Name": contact["Last Name"],

        "Company": contact["Company"],

        "Email": email_address,

        "Introduction Date": contact["Introduction Date"].strftime(
            "%Y-%m-%d"
        ),

       # "Recruiter": contact["RECRUITER_NAME"],

      #  "Recruiter Email": contact["Recruiter Email"],

        "Original Subject": original_subject,

        "Suggested Email Subject": email_subject,

        "Suggested Email Reply": email_reply,

        "LinkedIn Note": linkedin_note,

        "LinkedIn Search": linkedin_search,

        #"Original Email Excerpt": contact["Original Email Excerpt"],

        "Review Needed": "Yes",

        "Outreach Status": "Not Sent"

    })


df = pd.DataFrame(rows)

print(f"Final contact count: {len(df)}")

df.head()

In [ ]:
#Refactored code 

import os
from datetime import datetime

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment


# --------------------------------------------------
# 1. Define filename and check whether Excel has it open
# --------------------------------------------------

today = datetime.now().strftime("%Y-%m-%d")
OUTPUT_FILE = f"Recruiter_Introduction_Tracker_{today}.xlsx"
full_path = os.path.abspath(OUTPUT_FILE)

# Excel creates a temporary lock file when a workbook is open
lock_file = os.path.join(
    os.path.dirname(full_path),
    "~$" + os.path.basename(full_path)
)

if os.path.exists(lock_file):
    raise RuntimeError(
        f"File '{OUTPUT_FILE}' is open. Close the file and run again."
    )

# Delete the previous version if it exists
if os.path.exists(full_path):
    os.remove(full_path)


# --------------------------------------------------
# 2. Export DataFrame and load workbook
# --------------------------------------------------

df.to_excel(
    full_path,
    index=False,
    sheet_name="Recruiter Introductions"
)

wb = load_workbook(full_path)
ws = wb["Recruiter Introductions"]


# --------------------------------------------------
# 3. Format header row
# --------------------------------------------------

header_fill = PatternFill(
    start_color="17425E",
    end_color="17425E",
    fill_type="solid"
)

header_font = Font(
    color="FFFFFF",
    bold=True
)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(vertical="center", wrap_text=True)


# --------------------------------------------------
# 4. Set column widths
# --------------------------------------------------

column_widths = {
    "A": 16,
    "B": 18,
    "C": 22,
    "D": 32,
    "E": 18,
    "F": 16,
    "G": 32,
    "H": 35,
    "I": 35,
    "J": 65,
    "K": 55,
    "L": 25,
    "M": 65,
    "N": 16,
    "O": 18
}

for column, width in column_widths.items():
    ws.column_dimensions[column].width = width


# --------------------------------------------------
# 5. Format LinkedIn search links (Column L)
# --------------------------------------------------

linkedin_font = Font(
    color="0563C1",
    underline="single"
)

for row in range(2, ws.max_row + 1):
    cell = ws.cell(row=row, column=12)

    if cell.value:
        cell.hyperlink = str(cell.value)
        cell.value = "Search LinkedIn"
        cell.font = linkedin_font


# --------------------------------------------------
# 6. Format worksheet layout
# --------------------------------------------------

# Wrap text and align all data cells to the top
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )

# Freeze header and enable filters
ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions
ws.row_dimensions[1].height = 32


# --------------------------------------------------
# 7. Save finished workbook
# --------------------------------------------------

wb.save(full_path)

print(f"Excel tracker successfully saved: {OUTPUT_FILE}")



In [ ]:
pip install --upgrade jupyterlab-git